# KET-QA Retriever with Sentence-Transformers

논문 구현: 전처리, serialization 검증, negative sampling 확인, sentence-transformers 활용.

## 1. 경로 및 프로젝트 설정

In [1]:
import sys
import os
import json
import random
from collections import defaultdict

# 프로젝트 루트 (nb가 kb-adaptive 안에 있다고 가정)
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if "kb-adaptive" not in os.getcwd() else os.getcwd()
sys.path.insert(0, ROOT)

DATA_ROOT = os.path.join(ROOT, "dataset_ketqa")
TABLE_DIR = os.path.join(DATA_ROOT, "tables")
ENTITY_BASE_DIR = os.path.join(DATA_ROOT, "entity_base")
QA_DATA_DIR = os.path.join(DATA_ROOT, "data")

print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("train.json exists:", os.path.isfile(os.path.join(QA_DATA_DIR, "train.json")))
print("tables count:", len([f for f in os.listdir(TABLE_DIR) if f.endswith(".json")]) if os.path.isdir(TABLE_DIR) else 0)

ROOT: /Users/yek/Workspace/kb-adaptive
DATA_ROOT: /Users/yek/Workspace/kb-adaptive/dataset_ketqa
train.json exists: True
tables count: 15314


## 2. QA 데이터 로드 및 구조 시각화

In [3]:
from retriever.dataset import load_qa_split

# 소량만 로드 (전체는 무거우므로)
N_SAMPLE = 100
qa_all = load_qa_split(QA_DATA_DIR, "train")
qa_items = [x for x in qa_all if x.get("table_id")][:N_SAMPLE]
print(f"Loaded {len(qa_items)} QA items (with table_id)")
print("Keys of one item:", list(qa_items[0].keys()))

Loaded 100 QA items (with table_id)
Keys of one item: ['question_id', 'question', 'answer', 'whether_can_be_answerd', 'need_table_title', 'selected_answer_candidate', 'selected_evidence_candidate', 'answer_source', 'calculation_type', 'table_id']


In [5]:
!pip install pandas

In [6]:
import pandas as pd

# 첫 3개 항목 표로 보기
rows = []
for i, item in enumerate(qa_items[:3]):
    rows.append({
        "idx": i,
        "question_id": item.get("question_id", "")[:12] + "..",
        "question": (item.get("question", "")[:50] + "..") if len(item.get("question", "")) > 50 else item.get("question", ""),
        "table_id": item.get("table_id", ""),
        "num_evidence": len(item.get("selected_evidence_candidate", [])),
    })
pd.DataFrame(rows)

,idx,question_id,question,table_id,num_evidence
0,0,256e13557b53..,What is the attendance of the event whose venu...,Jordan_Brand_Classic_0,1
1,1,2aae69deeef3..,what is the year of the film by the director b...,Gene_Tierney_1,2
2,2,4a53c00e9208..,Who was the original club of the player who wa...,2001_AFL_Draft_0,1


## 3. Serialization 검증 (테이블 → T*, Triple → t*)

한 QA에 대해: 테이블 로드 → 엔티티 수집 → 서브그래프 triple 생성 → T*_sub, t* 문자열 확인.

In [7]:
from retriever.dataset import load_table, get_table_entity_ids
from retriever.kb import build_subgraph_triples, evidence_to_triple_dict
from retriever import serialization as ser

# 첫 번째 QA
item = qa_items[0]
table_id = item["table_id"]
table_obj = load_table(TABLE_DIR, table_id)
if not table_obj:
    raise FileNotFoundError(f"Table {table_id} not found in {TABLE_DIR}")

header = table_obj.get("header", [])
data = table_obj.get("data", [])
entity_ids = get_table_entity_ids(data)
triples = build_subgraph_triples(entity_ids, ENTITY_BASE_DIR)
triples = triples[:10]  # 처음 10개만

print("Question:", item["question"])
print("Table id:", table_id)
print("Header (first 5):", header[:5])
print("Num rows:", len(data))
print("Entity IDs (first 5):", list(entity_ids)[:5])
print("Num triples (capped 10):", len(triples))

Question: What is the attendance of the event whose venue has a capacity of 17,950 ?
Table id: Jordan_Brand_Classic_0
Header (first 5): [['Year', []], ['Result', []], ['Venue', []], ['City', []], ['Attendance', []]]
Num rows: 18
Entity IDs (first 5): ['Q16565', 'Q668676', 'Q984542', 'Q2985114', 'Q61']
Num triples (capped 10): 10


In [8]:
triples

[{'head_id': 'Q16565',
  'head_label': 'Charlotte',
  'rel_label': 'country',
  'tail_id': 'Q30',
  'tail_label': 'United States of America',
  'serialized': '[HEAD] Charlotte [REL] country [TAIL] United States of America',
  'is_relation': True},
 {'head_id': 'Q16565',
  'head_label': 'Charlotte',
  'rel_label': 'Commons category',
  'tail_id': None,
  'tail_label': 'Charlotte, North Carolina',
  'serialized': '[HEAD] Charlotte [REL] Commons category [TAIL] Charlotte, North Carolina',
  'is_relation': False},
 {'head_id': 'Q16565',
  'head_label': 'Charlotte',
  'rel_label': 'located in the administrative territorial entity',
  'tail_id': 'Q507770',
  'tail_label': 'Mecklenburg County',
  'serialized': '[HEAD] Charlotte [REL] located in the administrative territorial entity [TAIL] Mecklenburg County',
  'is_relation': True},
 {'head_id': 'Q16565',
  'head_label': 'Charlotte',
  'rel_label': 'twinned administrative body',
  'tail_id': 'Q159273',
  'tail_label': 'Arequipa',
  'serialize

In [ ]:
# Triple serialization (t*) 시각화: relational vs attribute
for i, t in enumerate(triples[:5]):
    print(f"--- Triple {i+1} ---")
    print("  head_id:", t["head_id"], "| head_label:", t["head_label"])
    print("  rel:", t["rel_label"], "| tail_id:", t["tail_id"], "| tail_label:", str(t["tail_label"])[:40])
    print("  serialized (t*):", t["serialized"][:100] + ("..." if len(t["serialized"]) > 100 else ""))
    print()

In [ ]:
# 서브테이블 직렬화 T*_sub (한 triple의 head entity 기준)
t0 = triples[0]
T_star = ser.table_to_serialized_with_subtable(header, data, t0["head_id"])
context = T_star + " " + t0["serialized"]
print("T*_sub (first 400 chars):")
print(T_star[:400] + "..." if len(T_star) > 400 else T_star)
print()
print("Context = T*_sub + t* (first 500 chars):")
print(context[:500] + "..." if len(context) > 500 else context)

In [ ]:
# Gold evidence → serialized triple 매칭 확인
gold_evidence = item.get("selected_evidence_candidate", [])
gold_triples = set()
for ev in gold_evidence:
    td = evidence_to_triple_dict(ev)
    if td:
        gold_triples.add(td["serialized"])
        print("Gold (In KB) serialized:", td["serialized"])

print("\nTriples that match gold (label=1):")
for t in triples:
    if t["serialized"] in gold_triples:
        print("  ", t["serialized"][:80] + "...")

## 4. (Question, Context, Label) 플랫 리스트 및 그룹 구성

RetrievalDataset과 동일한 방식으로 샘플 빌드 → (question, table)별 pos/neg 개수 시각화.

In [ ]:
from retriever.dataset import RetrievalDataset

# 소량 QA로 데이터셋 빌드 (캐시는 data_root/.retrieval_cache 에 저장됨)
ds = RetrievalDataset(
    qa_items,
    TABLE_DIR,
    ENTITY_BASE_DIR,
    split="train",
    n_negatives=25,
    max_triples_per_table=2000,
)
# 플랫 리스트: (question, context, label) per triple
samples = ds._samples
print("Total samples (q, context, label):", len(samples))
print("Sample keys:", list(samples[0].keys()) if samples else [])

In [ ]:
# 플랫 리스트를 DataFrame으로
flat = []
for s in samples:
    flat.append({
        "question": s["question"][:60] + ".." if len(s["question"]) > 60 else s["question"],
        "context_len": len(s["context"]),
        "label": s["label"],
        "table_id": s.get("table_id", ""),
    })
df = pd.DataFrame(flat)
print("Label distribution:")
print(df["label"].value_counts())
df.head(10)

In [ ]:
# (question, table_id) 그룹별 pos/neg 개수 (RetrievalDataset과 동일 키)
group_key = lambda s: (s["question"], s["table_id"])
groups = defaultdict(lambda: {"pos": [], "neg": []})
for s in samples:
    k = group_key(s)
    if s["label"] == 1:
        groups[k]["pos"].append(s)
    else:
        groups[k]["neg"].append(s)

group_stats = []
for k, v in list(groups.items())[:20]:
    group_stats.append({"question": k[0][:25] + "..", "table_id": k[1][:20], "n_pos": len(v["pos"]), "n_neg": len(v["neg"])})
pd.DataFrame(group_stats)

In [ ]:
import matplotlib.pyplot as plt

n_pos = [len(v["pos"]) for v in groups.values()]
n_neg = [len(v["neg"]) for v in groups.values()]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(n_pos, bins=range(0, max(n_pos)+2), edgecolor="black", alpha=0.7)
axes[0].set_title("Positives per (question, table)")
axes[0].set_xlabel("Count")
axes[1].hist(n_neg, bins=min(30, max(n_neg)+1), edgecolor="black", alpha=0.7)
axes[1].set_title("Negatives per (question, table)")
axes[1].set_xlabel("Count")
plt.tight_layout()
plt.show()

## 5. Negative Sampling 확인

한 그룹에서 1 pos + n_neg 개의 negative를 랜덤 샘플링해 (question, pos_context, neg_contexts) 형태로 확인.

In [ ]:
N_NEG = 5
# pos가 있고 neg가 충분한 그룹 하나 선택
candidates = [(k, v) for k, v in groups.items() if len(v["pos"]) >= 1 and len(v["neg"]) >= N_NEG]
key, group = random.choice(candidates) if candidates else (list(groups.items())[0])
pos_list = group["pos"]
neg_list = group["neg"]
pos_one = random.choice(pos_list)
negs_sampled = random.sample(neg_list, min(N_NEG, len(neg_list)))

print("Question:", pos_one["question"])
print("Table:", key[1])
print("\n--- Positive (1) ---")
print("Context (first 300 chars):", pos_one["context"][:300] + "..." if len(pos_one["context"]) > 300 else pos_one["context"])
print("\n--- Negatives (sampled", len(negs_sampled), ") ---")
for i, n in enumerate(negs_sampled):
    print(f"  [{i+1}] {n['context'][:120]}..." if len(n["context"]) > 120 else f"  [{i+1}] {n['context']}")

In [ ]:
# Negative sampling 결과를 표로 정리
neg_table = []
neg_table.append({"type": "Positive", "context_preview": pos_one["context"][:150] + "...", "label": 1})
for n in negs_sampled:
    neg_table.append({"type": "Negative", "context_preview": n["context"][:150] + "...", "label": 0})
pd.DataFrame(neg_table)

## 6. Serialization 저장/로드 검증

빌드한 samples를 JSON으로 저장 후 다시 로드해 동일한지 확인.

In [ ]:
# 디스크에 저장 (일부만)
serialized_sample = samples[:50]
out_path = os.path.join(ROOT, "outputs", "st_serialization_test.json")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(serialized_sample, f, ensure_ascii=False, indent=0)

# 다시 로드
with open(out_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)
print("Saved:", len(serialized_sample), "| Loaded:", len(loaded))
print("Match:", loaded[0]["question"] == serialized_sample[0]["question"] and loaded[0]["label"] == serialized_sample[0]["label"])
print("Path:", out_path)

## 7. Sentence-Transformers로 인코딩 및 유사도 시각화

Bi-encoder처럼 query와 context를 각각 인코딩한 뒤 코사인 유사도로 positive/negative 구분이 되는지 확인.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
print("Model loaded:", model_name)

In [ ]:
# 위에서 만든 1 pos + n neg 로 유사도 계산
query = pos_one["question"]
contexts = [pos_one["context"]] + [n["context"] for n in negs_sampled]
labels = [1] + [0] * len(negs_sampled)

q_emb = model.encode(query, convert_to_tensor=False)
c_embs = model.encode(contexts, convert_to_tensor=False)
# 코사인 유사도
similarities = np.dot(c_embs, q_emb) / (np.linalg.norm(c_embs, axis=1) * np.linalg.norm(q_emb) + 1e-9)
for i, (sim, lab) in enumerate(zip(similarities, labels)):
    print(f"  [{i}] {'POS' if lab == 1 else 'NEG'} sim={sim:.4f}")
print("\nPositive should have highest similarity:", np.argmax(similarities) == 0)

In [ ]:
# 유사도 막대 그래프
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(contexts))
colors = ["green" if l == 1 else "red" for l in labels]
ax.bar(x, similarities, color=colors, edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels([f"{'POS' if l==1 else 'NEG'}({i})" for i, l in enumerate(labels)])
ax.set_ylabel("Cosine similarity to query")
ax.set_title("Query–context similarity (1 pos + negatives)")
plt.tight_layout()
plt.show()

## 8. (선택) SentenceTransformers fine-tuning 예시

InputExample으로 (query, pos_context) / (query, neg_context) 쌍을 주고 `CrossEncoder` 또는 `SentenceTransformer` fit으로 학습할 수 있음. 여기서는 데이터 포맷만 준비.

In [ ]:
# MultipleNegativesRankingLoss: (anchor=query, positive=gold context). 배치 내 다른 샘플이 negative로 사용됨.
from sentence_transformers import InputExample

train_examples = []
for _key, grp in list(groups.items())[:30]:
    if not grp["pos"]:
        continue
    q = grp["pos"][0]["question"]
    pos_ctx = grp["pos"][0]["context"]
    train_examples.append(InputExample(texts=[q, pos_ctx]))

print("Prepared", len(train_examples), "training examples (anchor=query, positive=context)")
print("Example[0] query:", train_examples[0].texts[0][:80] + "...")
print("Example[0] positive:", train_examples[0].texts[1][:80] + "...")

---

**정리**: 전처리·직렬화(T*/t*)·그룹별 pos/neg·negative 샘플링·직렬화 저장/로드·sentence-transformers 유사도 시각화까지 확인. 실제 학습은 `train_bi_encoder.py` 또는 `SentenceTransformer.fit()` + `MultipleNegativesRankingLoss`로 진행 가능.